# Track 1 · 09 REVE + LoRA — done properly, against EEGNet

**One job.** Fine-tune REVE on the same THINGS-EEG2 windows EEGNet trained on, and see
whether it beats EEGNet: **8.92%** top-5 at 2,000 windows per person, **10.70%** at 6,000
(warm-up leaderboard 10.82%).

Self-contained: upload this one file. Needs `track1_08_things_prepare` to have run.

---

## What is different from the REVE runs that sat at chance

| | before | now |
|---|---|---|
| data | Alljoined, weak single-trial signal | THINGS, prepared with the harness's own pipeline |
| loss | tau 1, prediction unnormalised | **tau 0.07, both sides normalised** (the EEGNet recipe) |
| head | electrodes averaged, or one shared map | **each electrode has its own weights** |
| training | frozen, or LoRA with the wrong loss | LoRA on the 44 attention projections |

## The model

- REVE's pretrained weights frozen; rank-8 LoRA beside each attention projection.
- Head: each electrode's 512-d token is shrunk to 32 by one shared layer, then a final
  layer with separate weights per electrode maps to the 1536-d target. ~3.1M parameters.
- Input: the harness windows (120 Hz), resampled to REVE's 200 Hz and standardised per
  window inside the model.

The model class lives in `solvers/track1_image_reve_lora_perchan.py` and this notebook
trains that exact class, so the submission cannot drift from what was trained.

## Regularisation, aimed at the EEGNet sweet spot

Head dropout 0.3; LoRA rank 8 (rank 16 memorised before); adapters at lr 1e-4 and head
at 1e-3; the first epoch trains the head only, so early noisy gradients do not disturb
the pretrained encoder; no weight decay (it was inert at these rates); best validation
epoch kept.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

# Rebuild mode: package saved weights without retraining (e.g. after a Colab
# disconnect). Set RUN_TRAINING = False, REBUILD_TAG to the saved run's tag, and
# REBUILD_EXPECT_TOP5 to that run's test top-5, then Run all.
RUN_TRAINING = True
REBUILD_TAG = None              # e.g. 'reve_things6000_r8_s16_d0.4'
REBUILD_EXPECT_TOP5 = None      # e.g. 12.75; the rebuild must reproduce it

THINGS_TRAIN_WINDOWS = 'all'    # 2000 (windows/), 6000 (windows_6000/), or 'all': the whole
                                # session, each window once (track1_08 with 'all')

EPOCHS = 20
HEAD_ONLY_EPOCHS = 1            # adapters stay frozen for this many epochs
BATCH = 64
LR_ADAPTER, LR_HEAD = 1e-4, 1e-3
LORA_RANK, LORA_ALPHA = 8, 16
SHRINK = 32                     # per-electrode width before the final layer
DROPOUT = 0.3
TEMPERATURE = 0.07
USE_AMP = True                  # mixed precision on GPU; roughly halves the epoch time

MAKE_ZIP = True
VAL_IMAGE_FRAC = 0.2
SEED = 33

EEGNET = {2000: dict(top5=8.92, agg=15.00), 6000: dict(top5=10.70, agg=18.00)}
RUN_TAG = f'reve_things{THINGS_TRAIN_WINDOWS}_r{LORA_RANK}_s{SHRINK}_d{DROPOUT}'
print(f'NOTEBOOK VERSION: reve 2   settings {RUN_TAG}   epochs {EPOCHS}   '
      f'chance loss will be {__import__("math").log(BATCH):.4f}')

In [ ]:
%pip install -q 'braindecode>=1.8' huggingface_hub safetensors

import importlib
for mod in ('braindecode', 'torch'):
    print(f'{mod:12} {getattr(importlib.import_module(mod), "__version__", "ok")}')

In [ ]:
import json, os, shutil, sys, time, types, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
TH = BASE / 'things_eeg2'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', '/content/hf')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = USE_AMP and DEVICE == 'cuda'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def l2(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def _truth(y_ids):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    return gal, np.array([pos[y] for y in y_ids])


def rank_pct(pred, y_ids, emb):
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    return float((sims < sims[np.arange(len(sims)), truth][:, None]).mean())


def top5(pred, y_ids, emb):
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    k = min(5, len(gal))
    tk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, tk)]), 100.0 * k / len(gal)


def rank_null(pred, y_ids, n=200):
    """Validation rank under permuted labels, for these exact predictions."""
    gal, truth = _truth(y_ids)
    sims = l2(pred) @ l2(emb[gal]).T
    rows = np.arange(len(sims))
    return np.array([float((sims < sims[rows, p][:, None]).mean())
                     for p in (rng.permutation(truth) for _ in range(n))])


print(f'device {DEVICE}, mixed precision {AMP}')

## Step 1 — the THINGS windows and targets

In [ ]:
t = np.load(TH / 'image_embeddings.npz', allow_pickle=True)
emb = l2(t['emb'].astype('float32'))
DIM = emb.shape[1]
key = {k: i for i, k in enumerate(t['keys'].tolist())}

def things_train_files(TH, which):
    """(file, positions to skip) for one prepared training set. 'all' is the whole
    session, each window once: the earlier samples plus windows_rest/, minus the
    positions track1_08 found duplicated across them."""
    if which != 'all':
        d = TH / ('windows' if which == 2000 else f'windows_{which}')
        return [(f, set()) for f in sorted(d.glob('*_train.npz'))]
    out = []
    for rest in sorted((TH / 'windows_rest').glob('*_train.npz')):
        z = np.load(rest, allow_pickle=True)
        skip = json.loads(str(z['skip']))
        for folder in json.loads(str(z['sources'])):
            out.append((TH / folder / rest.name, set(skip.get(folder, []))))
        out.append((rest, set()))
    return out


parts = {'train': [[], [], []], 'test': [[], [], []]}
CH = None
for f, skip in (things_train_files(TH, THINGS_TRAIN_WINDOWS)
                + [(f, set()) for f in sorted((TH / 'windows').glob('*_test.npz'))]):
    z = np.load(f, allow_pickle=True)
    idx = np.array([key.get(k_, -1) for k_ in z['keys'].tolist()])
    k = (idx >= 0) & ~np.isin(np.arange(len(idx)), sorted(skip))
    sp = str(z['split'])
    parts[sp][0].append(z['X'][k])                   # stays float16 until batched
    parts[sp][1].append(idx[k])
    parts[sp][2].append(np.full(int(k.sum()), str(int(z['subject']))))
    CH = list(z['channels'])
if not parts['train'][0]:
    raise FileNotFoundError(f'no training windows for THINGS_TRAIN_WINDOWS = '
                            f'{THINGS_TRAIN_WINDOWS!r}; run track1_08 with that setting first')
Wtr = [np.concatenate(a) for a in parts['train']]
Wte = [np.concatenate(a) for a in parts['test']]
IN_SFREQ = float(json.loads((TH / 'manifest.json').read_text())['sfreq'])

val_imgs = set(rng.choice(np.unique(Wtr[1]), int(len(np.unique(Wtr[1])) * VAL_IMAGE_FRAC),
                          replace=False).tolist())
IV = np.array([y in val_imgs for y in Wtr[1]])
print(f'train {Wtr[0].shape}   test {Wte[0].shape}   {len(CH)} channels at {IN_SFREQ:g} Hz')
print(f'{(~IV).sum():,} fit / {IV.sum():,} val windows, {len(np.unique(Wte[1]))} test images')

## Step 2 — the model, from the submission file

The submission file is executed here to get the model class, so training and inference
share one definition. REVE's pretrained encoder is loaded into it, and the load is
checked tensor by tensor: a silent failure would mean fine-tuning a random encoder.

In [ ]:
# Source of truth: solvers/track1_image_reve_lora_perchan.py in the repo.
SUBMISSION_PY = "\"\"\"Track 01 submission: REVE with LoRA adapters and a per-electrode head.\n\nThe encoder is `brain-bzh/reve-base`. Its pretrained weights are frozen; LoRA learns a\nsmall rank-r correction beside each of the 44 attention projections\n(`transformer.layers.*.0.{to_qkv,to_out}`), so no pretrained weight is overwritten.\n\nThe head gives every electrode its own weights. Averaging electrodes together, or\napplying one shared map to each, both lost the signal on this task, and each electrode\nneeding its own mapping was the consistent finding. Each electrode's 512-d token is\nshrunk to a few numbers by one shared layer, then a final layer with separate weights\nper electrode maps them to the target embedding. The shrink keeps that layer small.\n\nThat makes the head specific to the electrodes it was trained on. Inputs are matched to\nthem by name: extra electrodes are ignored and missing ones are filled with zeros, which\nafter the harness's scaling means \"no information\". Electrodes REVE's position bank does\nnot know are left out, at training and at inference alike.\n\nInput is the harness's own preprocessing (120 Hz, RobustScaler, baseline, clamp),\nresampled to REVE's 200 Hz and standardised per window inside the model, so the\ntraining windows and the scoring windows go through identical code.\n\nThis file is also the model definition the training notebook uses, so the two cannot\ndrift apart.\n\"\"\"\n\nimport json\nimport os\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import REVE\n\nREVE_SFREQ = 200.0     # REVE's pretraining rate\nPATCH = 200            # REVE's patch_size in samples\nCLIP_SD = 15.0\nLORA_TARGETS = (\"to_qkv\", \"to_out\")\n\n\ndef resample_fft(x, n_out):\n    \"\"\"Resample along time via the rfft spectrum. Pure torch, since nothing is\n    installed at evaluation and linear interpolation aliases.\"\"\"\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass LoRALinear(nn.Module):\n    \"\"\"Frozen base plus a rank-r update. B starts at zero, so training begins at\n    exactly the pretrained function.\"\"\"\n\n    def __init__(self, base, r, alpha):\n        super().__init__()\n        self.base = base\n        for p in self.base.parameters():\n            p.requires_grad_(False)\n        self.A = nn.Parameter(torch.zeros(r, base.in_features))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r))\n        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)\n        self.scale = alpha / r\n\n    def forward(self, x):\n        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale\n\n\nclass PerElectrodeHead(nn.Module):\n    \"\"\"Shared shrink per electrode, then separate weights per electrode.\"\"\"\n\n    def __init__(self, n_chans, embed_dim, shrink, n_outputs, dropout):\n        super().__init__()\n        self.norm = nn.LayerNorm(embed_dim)\n        self.shrink = nn.Linear(embed_dim, shrink)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(n_chans * shrink, n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))\n        return self.out(z.flatten(1))\n\n\nclass RevePerChannel(nn.Module):\n\n    def __init__(self, channels, n_outputs, in_sfreq, rank=8, alpha=16, shrink=32,\n                 dropout=0.3, encoder_state=None):\n        super().__init__()\n        self.in_sfreq = float(in_sfreq)\n        self.net = REVE(\n            n_outputs=n_outputs, n_chans=len(channels), n_times=PATCH,\n            sfreq=REVE_SFREQ, chs_info=[{\"ch_name\": c} for c in channels],\n        )\n        self.net.final_layer = nn.Identity()\n        if encoder_state is not None:             # training: pretrained weights first\n            self.net.load_state_dict(encoder_state, strict=False)\n        for p in self.net.parameters():\n            p.requires_grad_(False)\n        names = [\n            n for n, m in self.net.named_modules()\n            if isinstance(m, nn.Linear) and n.split(\".\")[-1] in LORA_TARGETS\n        ]\n        for name in names:\n            parent = self.net.get_submodule(name.rsplit(\".\", 1)[0])\n            setattr(parent, name.split(\".\")[-1],\n                    LoRALinear(self.net.get_submodule(name), rank, alpha))\n        self.n_lora = len(names)\n\n        # The position bank returns only names it recognises and merely logs the\n        # shortfall, which later breaks the forward. Resolve here, identically for\n        # training and inference, and leave unknown electrodes out.\n        bank = self.net._position_bank.mapping\n        folded = {k.casefold(): k for k in bank}\n        keep, resolved = [], []\n        for i, ch in enumerate(channels):\n            name = ch if ch in bank else folded.get(ch.casefold())\n            if name is not None:\n                keep.append(i)\n                resolved.append(name)\n        if not keep:\n            raise RuntimeError(f\"no electrode resolved in REVE's bank: {channels[:8]}\")\n        self.used = resolved\n        self.register_buffer(\"keep_idx\", torch.tensor(keep), persistent=False)\n        self.register_buffer(\"pos\", self.net.get_positions(resolved), persistent=False)\n        self.head = PerElectrodeHead(len(keep), self.net.embed_dim, shrink,\n                                     n_outputs, dropout)\n\n    def adapter_parameters(self):\n        return [p for n, p in self.named_parameters() if n.endswith((\".A\", \".B\"))]\n\n    def forward(self, X):                          # (B, len(channels), T) at in_sfreq\n        X = X.float().index_select(1, self.keep_idx)\n        n_out = max(int(round(X.shape[-1] * REVE_SFREQ / self.in_sfreq)), PATCH)\n        X = resample_fft(X, n_out)\n        X = (X - X.mean(-1, keepdim=True)) / (X.std(-1, keepdim=True) + 1e-8)\n        X = torch.clamp(X, -CLIP_SD, CLIP_SD)\n        pos = self.pos.unsqueeze(0).expand(X.shape[0], -1, -1)\n        feats = self.net(X, pos=pos, return_features=True)[\"features\"]\n        return self.head(feats.mean(dim=2))       # average the time patches\n\n\nclass NameMatched:\n    \"\"\"Reorders incoming electrodes into the trained order, zero-filling gaps.\"\"\"\n\n    def __init__(self, model, channels, in_names, device):\n        self.model, self.device = model, device\n        lookup = {n.casefold(): i for i, n in enumerate(in_names)}\n        self.src = [lookup.get(c.casefold(), -1) for c in channels]\n        missing = [c for c, i in zip(channels, self.src) if i < 0]\n        print(\n            f\"[REVE-LoRA] matched {len(channels) - len(missing)}/{len(channels)} \"\n            f\"training channels by name among {len(in_names)} inputs; \"\n            f\"zero-filled: {missing if missing else 'none'}; REVE uses \"\n            f\"{len(model.used)} with known positions\"\n        )\n\n    @torch.no_grad()\n    def predict(self, X):\n        self.model.eval()\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        return self.model(out)\n\n\nclass Solver(CompetSolver):\n\n    name = \"REVE-LoRA-per-electrode\"\n\n    requirements = [\"pip::braindecode\"]\n\n    def load_model(self, meta):\n        sub = meta[\"submission_dir\"]\n        os.environ[\"REVE_POSITIONS_PATH\"] = str(sub)   # bundled; no download at eval\n        cfg = json.loads((sub / \"reve_config.json\").read_text())\n        model = RevePerChannel(\n            cfg[\"channels\"], cfg[\"n_outputs\"], meta[\"sfreq\"], rank=cfg[\"rank\"],\n            alpha=cfg[\"alpha\"], shrink=cfg[\"shrink\"], dropout=cfg[\"dropout\"],\n        )\n        state = torch.load(sub / \"weights.pt\", map_location=meta[\"device\"],\n                           weights_only=True)\n        missing, unexpected = model.load_state_dict(state, strict=False)\n        if unexpected or missing:\n            raise RuntimeError(f\"weights mismatch: missing {missing[:4]}, \"\n                               f\"unexpected {unexpected[:4]}\")\n        model = model.to(meta[\"device\"]).eval()\n        in_names = meta.get(\"ch_names\") or [c[\"ch_name\"] for c in meta[\"chs_info\"]]\n        return NameMatched(model, cfg[\"channels\"], in_names, meta[\"device\"])\n"

stub = types.ModuleType('benchmark_utils.base_solver')
stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
sys.modules['benchmark_utils.base_solver'] = stub
ns = {}
exec(compile(SUBMISSION_PY, 'submission.py', 'exec'), ns)


In [ ]:
if RUN_TRAINING:
    from braindecode.models import REVE
    enc = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=DIM, n_chans=len(CH),
                               n_times=200, sfreq=200.0, chs_info=[{'ch_name': c} for c in CH])
    enc_state = {k: v for k, v in enc.state_dict().items() if not k.startswith('final_layer')}
    del enc

    model = ns['RevePerChannel'](CH, DIM, IN_SFREQ, rank=LORA_RANK, alpha=LORA_ALPHA,
                                 shrink=SHRINK, dropout=DROPOUT,
                                 encoder_state=enc_state).to(DEVICE)

    # Every pretrained tensor must now sit in the model, under its LoRA-wrapped name.
    msd = model.state_dict()
    def wrapped(k):
        for t_ in ('to_qkv', 'to_out'):
            k = k.replace(f'.{t_}.', f'.{t_}.base.')
        return 'net.' + k
    bad = [k for k, v in enc_state.items()
           if wrapped(k) not in msd or not torch.equal(msd[wrapped(k)].cpu(), v.cpu())]
    adapters = model.adapter_parameters()
    head_params = list(model.head.parameters())
    n_ad = sum(p.numel() for p in adapters)
    n_hd = sum(p.numel() for p in head_params)
    n_all = sum(p.numel() for p in model.parameters())
    print(f'pretrained tensors loaded: {len(enc_state) - len(bad)}/{len(enc_state)}'
          + (f'   MISMATCH: {bad[:4]}' if bad else ''))
    print(f'REVE uses {len(model.used)}/{len(CH)} electrodes with known positions; '
          f'LoRA on {model.n_lora} projections')
    print(f'trainable: adapters {n_ad / 1e6:.2f}M + head {n_hd / 1e6:.2f}M '
          f'of {n_all / 1e6:.1f}M total')
    assert not bad, 'pretrained encoder did not load; do not train'


## Step 3 — train

Watch the gap between train and validation loss. Validation loss rising while train loss
falls is overfitting: raise dropout or lower `SHRINK`. Both flat near chance means it is
not learning: check the load line above first.

In [ ]:
if RUN_TRAINING:
    EMB_T = torch.tensor(emb, device=DEVICE)


    def clip_loss(pred, idx):
        logits = (F.normalize(pred.float(), dim=-1)
                  @ F.normalize(EMB_T[idx], dim=-1).T) / TEMPERATURE
        same = idx[:, None] == idx[None, :]
        same.fill_diagonal_(False)
        logits = logits.masked_fill(same, -1e4)
        return F.cross_entropy(logits, torch.arange(len(pred), device=pred.device))


    @torch.no_grad()
    def infer(X, bs=256):
        model.eval()
        out = []
        for i in range(0, len(X), bs):
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                out.append(model(torch.from_numpy(X[i:i + bs]).to(DEVICE)).float())
        return torch.cat(out)


    Xf, yf = Wtr[0][~IV], Wtr[1][~IV]
    Xv, yv = Wtr[0][IV], Wtr[1][IV]
    yv_t = torch.tensor(yv.astype('int64'), device=DEVICE)
    opt = torch.optim.AdamW([{'params': adapters, 'lr': LR_ADAPTER},
                             {'params': head_params, 'lr': LR_HEAD}], weight_decay=0.0)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    trainable = {n for n, p in model.named_parameters() if n.endswith(('.A', '.B'))
                 or n.startswith('head.')}
    print(f'chance loss {np.log(BATCH):.4f}')

    best, best_state, hist = -1, None, []
    for ep in range(1, EPOCHS + 1):
        on = ep > HEAD_ONLY_EPOCHS
        for p in adapters:
            p.requires_grad_(on)
        model.train()
        perm = rng.permutation(len(Xf))
        tot, t0 = 0.0, time.time()
        for i in range(0, len(perm) - 7, BATCH):
            b = perm[i:i + BATCH]
            xb = torch.from_numpy(Xf[b]).to(DEVICE)
            yb = torch.from_numpy(yf[b].astype('int64')).to(DEVICE)
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
                pred = model(xb)
            loss = clip_loss(pred, yb)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += float(loss) * len(b)
        pv = infer(Xv)
        vl = float(np.mean([float(clip_loss(pv[i:i + BATCH], yv_t[i:i + BATCH]))
                            for i in range(0, len(pv) - 7, BATCH)]))
        r = rank_pct(pv.cpu().numpy(), yv, emb)
        hist.append((ep, tot / len(Xf), vl, r))
        if r > best:
            best = r
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                          if k in trainable}
        print(f'  epoch {ep:2d}{"" if on else " (head only)"}  train loss {tot / len(Xf):.4f}  '
              f'val loss {vl:.4f}  val rank {r:.4f}{"  *" if r == best else ""}  '
              f'{time.time() - t0:.0f}s')
    # Only the trained tensors (adapters, head) are restored; the frozen encoder is
    # already in place. Assigned so Jupyter does not print the long list of
    # encoder keys that were deliberately not part of best_state.
    _ = model.load_state_dict(best_state, strict=False)


## Step 4 — score it against EEGNet

In [ ]:
if RUN_TRAINING:
    best_ep = max(hist, key=lambda h: h[3])
    vl_min = min(hist, key=lambda h: h[2])
    print(f'best val rank {best_ep[3]:.4f} at epoch {best_ep[0]}; '
          f'lowest val loss {vl_min[2]:.4f} at epoch {vl_min[0]}')

    pred = infer(Wte[0]).cpu().numpy()
    r_te = rank_pct(pred, Wte[1], emb)
    a_te, ch = top5(pred, Wte[1], emb)
    agg = {}
    for p_, y_, s_ in zip(pred, Wte[1], Wte[2]):
        agg.setdefault((s_, y_), []).append(p_)
    ks = sorted(agg)
    a_agg, _ = top5(np.stack([np.mean(agg[k_], 0) for k_ in ks]), np.array([k_[1] for k_ in ks]), emb)
    e = EEGNET.get(THINGS_TRAIN_WINDOWS)
    print(f'\n  {"REVE + LoRA":<14} rank {r_te:.4f}   top-5 {a_te:5.2f}% (per-window, = warm-up metric)'
          f'   {a_agg:5.2f}% (subject-agg)   chance {ch:.2f}%')
    if e:
        print(f'  {"EEGNet":<14}               top-5 {e["top5"]:5.2f}%'
              f'                                 {e["agg"]:5.2f}% (subject-agg)   same data')

    nullv = rank_null(infer(Xv).cpu().numpy(), yv)
    thr = float(np.percentile(nullv, 95))
    climb = best_ep[3] - hist[0][3]
    TRENDING = bool(best_ep[3] > thr and best_ep[0] > 1 and climb > 2 * nullv.std())
    print(f'\nval rank under permuted labels: mean {nullv.mean():.4f}  sd {nullv.std():.4f}  '
          f'95th {thr:.4f}; best {best_ep[3]:.4f} '
          f'({(best_ep[3] - nullv.mean()) / max(nullv.std(), 1e-9):.1f} sd above)')
    if TRENDING and e:
        verdict = 'beats' if a_te > e['top5'] else 'does not beat'
        print(f'Trending, and REVE {verdict} EEGNet on the same data '
              f'({a_te:.2f}% vs {e["top5"]:.2f}%).')
    elif TRENDING:
        print('Trending: it learns something that generalises to unseen images.')
    else:
        print('Not trending: validation rank never clears its own null.')

    torch.save(best_state, WEIGHTS / f'{RUN_TAG}_trainable.pt')
    print(f'saved -> {WEIGHTS / f"{RUN_TAG}_trainable.pt"} (adapters and head only)')


## Step 5 — package

Writes the full weights (REVE plus adapters plus head), the config and the submission
file, then loads the zip back the way the harness does and checks it gives **the same
predictions as the trained model**, and still runs with electrodes shuffled, extra or
missing. About 260 MB, mostly REVE's own weights.

In [ ]:
def package(model, tag, test_top5, rank, alpha, shrink, dropout):
    """Write weights, config and solver; reload them the way the harness does; zip
    only if the reloaded submission reproduces the model it came from."""
    import importlib.util
    out = BASE / 'submissions' / f'track1_{tag}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, out / 'weights.pt')
    (out / 'reve_config.json').write_text(json.dumps(dict(
        channels=CH, n_outputs=DIM, rank=rank, alpha=alpha, shrink=shrink,
        dropout=dropout, trained_on=f'THINGS-EEG2 ({tag})', test_top5=test_top5), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)
    bank = [p for p in (Path.home() / 'mne_data').glob('*reve*positions*.json')]
    if not bank:
        import huggingface_hub
        bank = [Path(huggingface_hub.hf_hub_download('brain-bzh/reve-positions',
                                                     'positions.json'))]
    shutil.copy(bank[0], out / 'reve_positions.json')

    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    def run(names, X):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                     'sfreq': IN_SFREQ, 'submission_dir': out,
                                     'device': DEVICE, 'n_outputs': DIM})
        return m.predict(torch.from_numpy(X.astype('float32'))).cpu().numpy()

    Xs = Wte[0][:8].astype('float32')
    base = run(CH, Xs)
    # Reference in full precision: the submission runs without autocast, so a
    # half-precision reference would only measure fp16 rounding.
    with torch.no_grad():
        model.eval()
        ref = model(torch.from_numpy(Xs).to(DEVICE)).float().cpu().numpy()
    perm = rng.permutation(len(CH))
    shuffled = run([CH[i] for i in perm], Xs[:, perm])
    wide = run(list(CH) + [f'X{i}' for i in range(31)],
               np.concatenate([Xs, rng.standard_normal((8, 31, Xs.shape[-1])).astype('float32')], 1))
    drop = [c for c in CH if c not in ('Oz', 'Fp1')]
    part = run(drop, Xs[:, [CH.index(c) for c in drop]])
    checks = [
        ('same as the model it came from', np.allclose(base, ref, atol=1e-3),
         f'max |diff| {np.abs(base - ref).max():.1e}'),
        ('order-independent', np.allclose(base, shuffled, atol=1e-4),
         f'max |diff| {np.abs(base - shuffled).max():.1e}'),
        ('extra electrodes', np.allclose(base, wide, atol=1e-4), 'ignored, output identical'),
        ('missing electrodes', part.shape == base.shape and np.isfinite(part).all(),
         'zero-filled, still runs'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if not all(c[1] for c in checks):
        print('\nNot zipping: a check failed.')
        return None
    zpath = shutil.make_archive(str(out), 'zip', out)
    print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e6:.0f} MB)')
    return zpath


if RUN_TRAINING and MAKE_ZIP:
    if TRENDING or globals().get('FORCE_ZIP'):
        package(model, RUN_TAG, a_te, LORA_RANK, LORA_ALPHA, SHRINK, DROPOUT)
    else:
        print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')


## Rebuild a submission from saved weights (no retraining)

Training saves only what it trained (`<tag>_trainable.pt`: LoRA adapters and head). This
rebuilds the full model from REVE's pretrained encoder plus those tensors, then
**re-scores it on the test windows and refuses to package unless it reproduces the
training run's top-5**. That guard matters: LoRA's `alpha` is not in the tag, and a
wrong value loads without error but gives a different model.

Set `RUN_TRAINING = False`, `REBUILD_TAG`, `REBUILD_EXPECT_TOP5` in the first cell, then
Run all. Training is skipped; only the data loading and this rebuild run.


In [ ]:
if REBUILD_TAG:
    import re
    from braindecode.models import REVE
    m_ = re.fullmatch(r'reve_things(\d+)_r(\d+)_s(\d+)_d([\d.]+)', REBUILD_TAG)
    if not m_:
        raise ValueError(f'unrecognised tag {REBUILD_TAG!r}')
    rank, shrink, dropout = int(m_[2]), int(m_[3]), float(m_[4])
    alpha = LORA_ALPHA                         # not in the tag; the score check below guards it
    path = WEIGHTS / f'{REBUILD_TAG}_trainable.pt'
    saved = torch.load(path, map_location='cpu', weights_only=True)

    enc = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=DIM, n_chans=len(CH),
                               n_times=200, sfreq=200.0, chs_info=[{'ch_name': c} for c in CH])
    enc_state = {k: v for k, v in enc.state_dict().items() if not k.startswith('final_layer')}
    del enc
    model = ns['RevePerChannel'](CH, DIM, IN_SFREQ, rank=rank, alpha=alpha, shrink=shrink,
                                 dropout=dropout, encoder_state=enc_state).to(DEVICE)
    res = model.load_state_dict(saved, strict=False)
    absent = [k for k in saved if k not in model.state_dict()]
    if res.unexpected_keys or absent:
        raise RuntimeError(f'saved tensors do not fit this model: {(res.unexpected_keys or absent)[:4]}')
    print(f'restored {len(saved)} trained tensors from {path.name} '
          f'(rank {rank}, alpha {alpha}, shrink {shrink}, dropout {dropout})')

    model.eval()
    with torch.no_grad():
        pred = torch.cat([model(torch.from_numpy(Wte[0][i:i + 256]).to(DEVICE)).float()
                          for i in range(0, len(Wte[0]), 256)]).cpu().numpy()
    a_re, ch = top5(pred, Wte[1], emb)
    print(f'rebuilt model: test top-5 {a_re:.2f}% (chance {ch:.2f}%)')
    if REBUILD_EXPECT_TOP5 is not None:
        if abs(a_re - REBUILD_EXPECT_TOP5) > 0.5:
            raise RuntimeError(f'does not reproduce the training run ({a_re:.2f}% vs '
                               f'{REBUILD_EXPECT_TOP5}%); check LORA_ALPHA and the tag')
        print(f'reproduces the training run ({REBUILD_EXPECT_TOP5}%), within half-precision rounding')
    package(model, REBUILD_TAG, a_re, rank, alpha, shrink, dropout)
